# Module 3 - Google OKF: let an agent navigate the knowledge base

No chunking, no embeddings, no index. The knowledge base is a folder of Markdown files with YAML frontmatter, and the
model gets four tools - the same verbs you would use in a terminal:

| Tool | Terminal analogue |
|---|---|
| `list_okf_topics` | `ls` + reading the frontmatter summaries |
| `grep_okf` | `grep -rn` |
| `read_okf_document` | `cat` (the **whole** file) |
| `follow_related_links` | opening the cross-links in the frontmatter |

You need a tool-calling LLM for the real agent (free OpenRouter key, or LM Studio with `FORMATION_MODEL=formation-local`).
Without one, the scripted navigator below still shows the same tools in action.

In [ ]:
import os, sys, json, asyncio, time, logging
from pathlib import Path
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)                                   # every path below is relative to the project root
os.environ.setdefault("HF_HOME", str(ROOT / "models" / "huggingface"))
logging.getLogger("neo4j").setLevel(logging.ERROR)
MODEL = os.getenv("FORMATION_MODEL", "formation-fast")   # use "formation-local" if you run LM Studio / Ollama

from backend.okf_agents import tools
from backend.okf_agents.deep_agent_harness import OKFDeepAgentHarness
from backend.okf_agents.pipeline import OKFAgentPipeline
from backend.graph_rag.pipeline import GraphRAGPipeline
from backend.graph_rag.neo4j_client import Neo4jGraphClient
from backend.graph_rag.qdrant_hybrid import QdrantHybridStore
print("tools:", [t["function"]["name"] for t in tools.TOOL_SCHEMAS])

## 1. The tools by hand

In [ ]:
topics = tools.list_okf_topics("hardware_manuals")
for t in topics[:3]: print(f"{t['doc_id']:36s} {t['type']:18s} {t['summary'][:70]}")
print()
for h in tools.grep_okf("hardware_manuals", "0x4A7B_VRM_OVERVOLT"): print(h["doc_id"], h["match_count"], "matches; first:", h["matches"][0]["text"][:90])
doc = tools.read_okf_document("hardware_manuals", "aor3_02_power_shelf_manual")
print("\nread whole document:", len(doc["content"]), "chars;", len(doc["images"]), "figures; related:", doc["related"])

## 2. Scripted navigation (no LLM): same tools, fixed plan

In [ ]:
nav = OKFDeepAgentHarness("hardware_manuals").scripted_navigation("What does fault code 0x8F11_TEMP_CRIT mean?")
for s in nav["trajectory"]: print(f"{s['step']}. {s['action']}({s['args']}) -> {s['observation']}")

## 3. The real agent: an LLM chooses the next tool call

In [ ]:
async def ask(q, corpus):
    r = await OKFAgentPipeline().execute_query(q, corpus=corpus, model_name=MODEL)
    print("agent_mode:", r["agent_mode"], "| answer_mode:", r["answer_mode"], "| latency:", r["latency_seconds"], "s")
    if r["llm_error"]: print("LLM problem:", r["llm_error"][:160])
    for s in r["agent_trajectory"]: print(f"  {s['step']}. {s['action']}({s['args']}) -> {s['observation']}")
    print("\n" + r["answer"][:900]); return r

r = await ask("Which fault codes are critical and show a solid red PWR LED in the Thermal subsystem?", "hardware_manuals")

## 4. Same question, both tracks

In [ ]:
graph = GraphRAGPipeline(Neo4jGraphClient(), QdrantHybridStore())
q = "Which monitoring and observability tools does Syone use, and which of them does it hold official partnerships for?"
a = await graph.execute_query(q, corpus="syone_corp", model_name=MODEL)
b = await OKFAgentPipeline().execute_query(q, corpus="syone_corp", model_name=MODEL)
for name, r in (("GraphRAG", a), ("OKF agent", b)):
    print(f"=== {name}: {r['latency_seconds']}s, {r['context_chars']:,} context chars, mode={r['answer_mode']}")
    print(r["answer"][:600], "\n")

## 5. Optional: the same idea with LangChain Deep Agents
`deepagents` adds planning, a virtual filesystem and sub-agents on top of the same tool-calling loop. Here we hand it our OKF tools.
(Needs a tool-calling model reachable through the gateway.)

In [ ]:
RUN_DEEPAGENTS = False
if RUN_DEEPAGENTS:
    from deepagents import create_deep_agent
    from langchain_openai import ChatOpenAI

    def list_okf_topics(corpus: str) -> list:
        """List all documents of an OKF corpus ('syone_corp' or 'hardware_manuals') with summaries."""
        return tools.run_tool(corpus, "list_okf_topics", {})
    def grep_okf(corpus: str, pattern: str) -> list:
        """Exact text search across all documents of the corpus."""
        return tools.grep_okf(corpus, pattern)
    def read_okf_document(corpus: str, doc_id: str) -> dict:
        """Read one whole document (frontmatter + body)."""
        return tools.read_okf_document(corpus, doc_id)

    from backend.llm import LITELLM_URL, LITELLM_MASTER_KEY          # gateway URL + key from .env
    llm = ChatOpenAI(model=MODEL, base_url=LITELLM_URL, api_key=LITELLM_MASTER_KEY)
    agent = create_deep_agent(model=llm, tools=[list_okf_topics, grep_okf, read_okf_document],
                              system_prompt="Answer only from the OKF documents; quote codes exactly.")
    out = agent.invoke({"messages": [{"role": "user", "content": "What does fault 0x10E4_ECC_MULTI_BIT require the technician to do? Corpus: hardware_manuals"}]})
    print(out["messages"][-1].content)
else:
    print("Set RUN_DEEPAGENTS = True to run it.")

## When to use which

| | GraphRAG + BM25 | OKF agent |
|---|---|---|
| Setup | Chunk, embed, extract entities, index | None: Markdown files are the index |
| Exact codes | Excellent (BM25) | Excellent (`grep`) |
| Multi-document questions | Needs the graph or luck in top-k | Natural: follow links, read whole documents |
| Cost per question | Low (a few chunks) | Higher (whole documents, several LLM calls) |
| Scales to | Millions of documents | Limited by context window and tool-call budget |
| Fails when | The needed chunk is not in top-k | The model does not find the right document |

Notebook 5 measures both on the same 34 questions.